# Generating Graphs

In [42]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import matplotlib.colors as mcolors
from matplotlib import colormaps
from itertools import cycle
from matplotlib.lines import Line2D
import os
import re

# Load Data

In [43]:
output_dir = "../graphs/v1_11_datasets_w_mteb"
os.makedirs(output_dir, exist_ok=True)

filepaths = [
    "../data/eval/results_eval_mteb_v1_11_datasets_w_mteb.csv",
    "../data/eval/results_eval_mteb_default.csv",
    "../data/eval/results_eval_mteb_closed_source.csv"
    ]

filepath_finetuned_matryoshka = "../data/eval/results_eval_mteb_v1_11_datasets_w_mteb.csv"
filepath_baseline = "../data/eval/results_eval_mteb_default.csv"
# model_names = [
#     "e5-large-matryoshka-sts-pt",
#     "Legal-BERTimbau-sts-large-ma-v3",
#     "Qwen3-Embedding-0.6B",
#     "multilingual-e5-large"
#     ]#['BERTimbau-large-matryoshka-sts-pt', 'BERTimbau-large-sts-pt'] 
# model_names = ['ModBERTBr-matryoshka-sts-pt', 'ModBERTBr-sts-pt', 'e5-large-matryoshka-sts-pt', 'e5-large-sts-pt', 'BERTimbau-large-matryoshka-sts-pt', 'BERTimbau-large-sts-pt'] 
model_names = []
model_names_to_remove = [
    'BERTimbau-large-sts-pt',
    'e5-large-sts-pt',
    'ModBERTBr-sts-pt',
    'bert-base-portuguese-cased',
    'mmBERT-base',
    'ModBERTBr',
    'bert-large-portuguese-cased',
    'NeoBERTugues'
    ]
# model_names_to_remove = ['ModBERTBr-sts-pt', 'e5-large-sts-pt', 'BERTimbau-large-sts-pt', 'bert-base-portuguese-cased']

results = [pd.read_csv(filepath) for filepath in filepaths]

# Pre-processing

In [44]:
task_type_map = {
    "Assin2STS": "STS",
    "SICK-BR-STS": "STS",
    "STSBenchmarkMultilingualSTS": "STS",
    'MassiveIntentClassification': "Classification",
    'MultiHateClassification': "Classification",
    'BrazilianToxicTweetsClassification': "Classification",
    'HateSpeechPortugueseClassification': "Classification",
    'TweetSentimentClassification': "Classification",
    'WikipediaRerankingMultilingual': "Reranking",
    'XGlueWPRReranking': "Reranking",
    'MultiLongDocReranking': "Reranking",
    'WebFAQRetrieval': "Retrieval",
    'WikipediaRetrievalMultilingual': "Retrieval",
    'MultiLongDocRetrieval': "Retrieval"
}

max_emb_dim_map = {
    'paraphrase-multilingual-MiniLM-L12-v2': 384,
    'BERTimbau': 1024,
    'Qwen3': 1024,
    'bert-base-portuguese-cased': 1024,
    'mmBERT': 768,
    'e5-large': 1024,
    'e5-small': 384,
    'e5-base': 768,
    'mpnet-base': 768,
    'ModBERTBr': 768,
    'NeoBERTugues': 768,
    'Qwen3-Embedding-0.6B-sts-pt': 1024,
    'serafim': 1536,
    'bert-large': 1024,
    'gte-multilingual': 768
}

model_size_map = {
    'BERTimbau': 335_000_000,
    'Qwen3': 600_000_000,
    'bert-base-portuguese-cased': 335_000_000,
    'mmBERT': 110_000_000,
    'e5-large': 550_000_000,
    'e5-small': 100_000_000,
    'e5-base': 270_000_000,
    'mpnet-base': 300_000_000,
    'ModBERTBr': 149_000_000,
    'NeoBERTugues': 110_000_000,
    'Qwen3-Embedding-0.6B-sts-pt': 600_000_000,
    'serafim': 900_000_000,
    'bert-large': 335_000_000,
    'gte-multilingual': 305_000_000
}

max_token_map = {
    'BERTimbau': 512,
    'Qwen3': 32_000,
    'mmbert-embed': 32_000,
    'bert-base-portuguese-cased': 512,
    'mmBERT-base': 8192,
    'e5-large': 512,
    'e5-small': 512,
    'e5-base': 512,
    'mpnet-base': 128,
    'ModBERTBr': 512,
    'NeoBERTugues': 8192,
    'Qwen3-Embedding-0.6B-sts-pt': 32_000,
    'serafim': 128,
    'bert-large': 512,
    'gte-multilingual': 8192
}

In [45]:
def map_model_name(model_raw_name: str, mapping: dict) -> int:
    matches = [
        value
        for key, value in mapping.items()
        if key.lower() in model_raw_name.lower()
    ]

    if len(matches) > 1:
        raise ValueError(
            f"Multiple matches found for '{model_raw_name}': "
            f"{[k for k in mapping if k.lower() in model_raw_name.lower()]}"
        )
    if len(matches) == 0:
        return None  # ou raise ValueError se quiser exigir match obrigatório

    return matches[0]

In [46]:
results_all = pd.concat(results, ignore_index=True)
results_all['task_type'] = results_all['task_name'].map(task_type_map)
results_all.dropna(subset=['task_type'], inplace=True)
results_all['main_score'] = results_all['main_score'] * 100
results_all['model_raw_name'] = results_all['model_name'].apply(lambda x: x.split("/")[-1])
results_all["max_emb_size"] = results_all["model_raw_name"].apply(
    lambda x: map_model_name(x, max_emb_dim_map)
)
results_all["model_size"] = results_all["model_raw_name"].apply(
    lambda x: map_model_name(x, model_size_map)
)
results_all["max_token"] = results_all["model_raw_name"].apply(
    lambda x: map_model_name(x, max_token_map)
)
results_all['embedding_dim_full'] = results_all['embedding_dim'].combine_first(results_all['max_emb_size']).astype(int)

if model_names:
    results_all = results_all.query("model_raw_name.isin(@model_names)")

if model_names_to_remove:
    results_all = results_all.query("~model_raw_name.isin(@model_names_to_remove)")

results_all.head()

,model_name,embedding_dim,task_name,main_score,task_key,task_type,model_raw_name,max_emb_size,model_size,max_token,embedding_dim_full
0,iara-project/e5-large-matryoshka-sts-pt,NaN,Assin2STS,84.103477,NaN,STS,e5-large-matryoshka-sts-pt,1024.0,550000000.0,512.0,1024
1,iara-project/e5-large-matryoshka-sts-pt,NaN,SICK-BR-STS,91.795896,NaN,STS,e5-large-matryoshka-sts-pt,1024.0,550000000.0,512.0,1024
2,iara-project/e5-large-matryoshka-sts-pt,NaN,STSBenchmarkMultilingualSTS,87.997523,NaN,STS,e5-large-matryoshka-sts-pt,1024.0,550000000.0,512.0,1024
3,iara-project/e5-large-matryoshka-sts-pt,NaN,MassiveIntentClassification,66.284236,NaN,Classification,e5-large-matryoshka-sts-pt,1024.0,550000000.0,512.0,1024
4,iara-project/e5-large-matryoshka-sts-pt,NaN,MultiHateClassification,66.670000,NaN,Classification,e5-large-matryoshka-sts-pt,1024.0,550000000.0,512.0,1024


In [47]:
# cond_e5 = results_all['model_raw_name'].str.contains('e5')
# cond_bertimbau = results_all['model_raw_name'].str.contains('BERTimbau')
# cond_qwen = results_all['model_raw_name'].str.contains('Qwen')
# cond_mmbert_embed = results_all['model_raw_name'].str.contains('mmbert-embed')
# results_all = results_all.loc[cond_e5 | cond_bertimbau | cond_qwen | cond_mmbert_embed]
results_all.head()

,model_name,embedding_dim,task_name,main_score,task_key,task_type,model_raw_name,max_emb_size,model_size,max_token,embedding_dim_full
0,iara-project/e5-large-matryoshka-sts-pt,NaN,Assin2STS,84.103477,NaN,STS,e5-large-matryoshka-sts-pt,1024.0,550000000.0,512.0,1024
1,iara-project/e5-large-matryoshka-sts-pt,NaN,SICK-BR-STS,91.795896,NaN,STS,e5-large-matryoshka-sts-pt,1024.0,550000000.0,512.0,1024
2,iara-project/e5-large-matryoshka-sts-pt,NaN,STSBenchmarkMultilingualSTS,87.997523,NaN,STS,e5-large-matryoshka-sts-pt,1024.0,550000000.0,512.0,1024
3,iara-project/e5-large-matryoshka-sts-pt,NaN,MassiveIntentClassification,66.284236,NaN,Classification,e5-large-matryoshka-sts-pt,1024.0,550000000.0,512.0,1024
4,iara-project/e5-large-matryoshka-sts-pt,NaN,MultiHateClassification,66.670000,NaN,Classification,e5-large-matryoshka-sts-pt,1024.0,550000000.0,512.0,1024


# Summarized Metrics

## Ratio Metrics

In [48]:
df_results_ratio = pd.DataFrame()

results_all["num_dims"] = results_all["embedding_dim"].fillna(results_all["embedding_dim_full"])
results_all["num_dims"] = pd.to_numeric(results_all["num_dims"], errors="coerce")
df_ratio = results_all.copy()

df_ratio["num_dims"] = pd.to_numeric(df_ratio["num_dims"], errors="coerce").astype(int)
df_ratio["main_score"] = pd.to_numeric(df_ratio["main_score"], errors="coerce")

df_ratio = df_ratio.dropna(subset=["task_type", "model_raw_name", "num_dims", "main_score"])

df_ratio = (
    df_ratio
    .groupby(["task_type", "model_raw_name", "num_dims"], as_index=False)["main_score"]
    .mean()
)

task_types = sorted(df_ratio["task_type"].dropna().unique())

for ttype in task_types:
    df_tt = df_ratio[df_ratio["task_type"] == ttype].copy()

    max_dims = (
        df_tt.groupby("model_raw_name")["num_dims"]
        .max()
        .rename("max_dim")
        .reset_index()
    )
    df_tt = df_tt.merge(max_dims, on="model_raw_name", how="left")

    df_max = df_tt[df_tt["num_dims"] == df_tt["max_dim"]][
        ["model_raw_name", "main_score"]
    ].rename(columns={"main_score": "max_score"})

    df_tt = df_tt.merge(df_max, on="model_raw_name", how="left")
    df_tt["ratio"] = df_tt["main_score"] / df_tt["max_score"]

    df_results_ratio = pd.concat([df_results_ratio, df_tt], axis=0, ignore_index=True)
df_results_ratio

,task_type,model_raw_name,num_dims,main_score,max_dim,max_score,ratio
0,Classification,BERTimbau-large-matryoshka-sts-pt,64,47.653795,1024,51.815073,0.919690
1,Classification,BERTimbau-large-matryoshka-sts-pt,128,49.445392,1024,51.815073,0.954267
2,Classification,BERTimbau-large-matryoshka-sts-pt,256,50.379274,1024,51.815073,0.972290
3,Classification,BERTimbau-large-matryoshka-sts-pt,512,51.299767,1024,51.815073,0.990055
4,Classification,BERTimbau-large-matryoshka-sts-pt,1024,51.815073,1024,51.815073,1.000000
...,...,...,...,...,...,...,...
303,STS,text-embedding-3-large,256,78.865044,3072,79.127511,0.996683
304,STS,text-embedding-3-large,512,79.066319,3072,79.127511,0.999227
305,STS,text-embedding-3-large,1024,79.134671,3072,79.127511,1.000090
306,STS,text-embedding-3-large,1536,79.159398,3072,79.127511,1.000403


In [49]:
df_pivot = df_results_ratio.pivot(
    index=["task_type", "model_raw_name"],
    columns="num_dims",
    values="ratio"
).reset_index().drop(columns=[768, 1024])
df_pivot['full_emb_dim'] = 1.0
df_pivot.to_csv("../data/eval/ablation_ratio.csv", index=False)

In [50]:
# 1) Extract a "base model" (remove the "-matryoshka" marker)
df_pivot["base_model"] = df_pivot["model_raw_name"].str.replace(
    "-matryoshka", "", regex=False
)

# 2) Flag whether it's matryoshka or not
df_pivot["is_matryoshka"] = df_pivot["model_raw_name"].str.contains("matryoshka")

# 3) Identify ratio columns (all numeric dim columns)
dim_cols = [c for c in df_pivot.columns if isinstance(c, int)]

# 4) Split into matryoshka and non-matryoshka
df_mat = df_pivot[df_pivot["is_matryoshka"]].copy()
df_non = df_pivot[~df_pivot["is_matryoshka"]].copy()

# 5) Merge pairs (same task_type + base_model)
df_diff = df_mat.merge(
    df_non,
    on=["task_type", "base_model"],
    suffixes=("_mat", "_non")
)

# 6) Compute signed differences: (matryoshka - non_matryoshka)
for col in dim_cols:
    df_diff[col] = df_diff[f"{col}_mat"] - df_diff[f"{col}_non"]

# 7) Keep only relevant columns
df_diff = df_diff[["task_type", "base_model"] + dim_cols]

# Optional: rename base_model back to model group name
df_diff = df_diff.rename(columns={"base_model": "model_group"})
df_diff.to_csv("../data/eval/ablation_ratio_diff.csv", index=False)

## Compare two models

In [ ]:
baseline_model = 'text-embedding-3-large'
compare_model = 'gte-multilingual-base'

In [12]:
results_max = results_all.sort_values(by = 'embedding_dim_full', ascending=False).groupby(['model_raw_name', 'task_name']).head(1)
grouped_results = results_max.query("model_raw_name.isin([@baseline_model, @compare_model])") \
                    .groupby(["model_raw_name", "task_type"])['main_score'] \
                    .mean() \
                    .reset_index()
grouped_results

,model_raw_name,task_type,main_score
0,gte-multilingual-base,Classification,0.502430
1,gte-multilingual-base,Reranking,0.841557
2,gte-multilingual-base,Retrieval,0.768990
3,gte-multilingual-base,STS,0.803772


In [13]:
def percent_diff_by_task_type(grouped_results, baseline_model, compare_model):
    wide = (
        grouped_results
        .pivot(index="task_type", columns="model_raw_name", values="main_score")
        .reindex(columns=[baseline_model, compare_model])
    )

    out = pd.DataFrame(index=wide.index)
    out["baseline"] = wide[baseline_model]
    out["compare"]  = wide[compare_model]
    out["abs_diff"] = out["compare"] - out["baseline"]

    # avoid division-by-zero issues
    out["pct_diff_vs_baseline"] = np.where(
        out["baseline"].ne(0),
        (out["abs_diff"] / out["baseline"]) * 100.0,
        np.nan
    )

    return out.reset_index().sort_values("task_type")

# usage
diff_table = percent_diff_by_task_type(grouped_results, baseline_model, compare_model)
diff_table

,task_type,baseline,compare,abs_diff,pct_diff_vs_baseline
0,Classification,NaN,0.502430,NaN,NaN
1,Reranking,NaN,0.841557,NaN,NaN
2,Retrieval,NaN,0.768990,NaN,NaN
3,STS,NaN,0.803772,NaN,NaN


In [122]:
print(compare_model)
df_compare_ratio = df_results_ratio.query("model_raw_name == @compare_model & num_dims == 64")
print(f"Média Ratio: {df_compare_ratio['ratio'].mean()}")
df_compare_ratio

serafim-900m-portuguese-pt-sentence-encoder
Média Ratio: 0.7954196255516123


,task_type,model_raw_name,num_dims,main_score,max_dim,max_score,ratio
67,Classification,serafim-900m-portuguese-pt-sentence-encoder,64,0.462750,1536,0.523372,0.884169
145,Reranking,serafim-900m-portuguese-pt-sentence-encoder,64,0.726602,1536,0.809840,0.897216
216,Retrieval,serafim-900m-portuguese-pt-sentence-encoder,64,0.244983,1536,0.548902,0.446316
289,STS,serafim-900m-portuguese-pt-sentence-encoder,64,0.827295,1536,0.867206,0.953978


In [123]:
print(baseline_model)
df_compare_ratio = df_results_ratio.query("model_raw_name == @baseline_model & num_dims == 64")
print(f"Média Ratio: {df_compare_ratio['ratio'].mean()}")
df_compare_ratio

bert-large-portuguese-cased
Média Ratio: 0.8508720559673806


,task_type,model_raw_name,num_dims,main_score,max_dim,max_score,ratio
28,Classification,bert-large-portuguese-cased,64,0.468901,1024,0.513786,0.912637
106,Reranking,bert-large-portuguese-cased,64,0.730032,1024,0.765435,0.953747
181,Retrieval,bert-large-portuguese-cased,64,0.186663,1024,0.317170,0.588528
250,STS,bert-large-portuguese-cased,64,0.598883,1024,0.631350,0.948576


# Export Graphs

## Dimension Truncation Metrics

### Graph General

In [41]:
# -------------------------
# Utils
# -------------------------
def sanitize_filename(name):
    name = str(name)
    name = re.sub(r"[^\w\s-]", "", name)
    name = re.sub(r"\s+", "_", name)
    return name.lower()


def apply_y_axis_formatting(
    ax,
    series,
    step=0.03,
    margin=0.01,
    y_bottom_override=None,  # force inferior y-limit if provided
    y_top_override=None,  # force inferior y-limit if provided
):
    """
    If y_bottom_override is set (e.g., 0.40), the y-axis lower bound will be forced
    to that value to create extra space at the bottom (e.g., for legends).
    """
    y_min = float(series.min())
    y_max = float(series.max())

    y_bottom = (np.floor(y_min / step) * step) - margin
    y_top = (np.floor(y_max / step) * step) + step + margin

    if y_bottom_override is not None:
        y_bottom = float(y_bottom_override)

    if y_top_override is not None:
        y_top = float(y_top_override)

    ax.set_ylim(y_bottom, y_top)
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.tick_params(axis="y", labelsize=25)
    ax.tick_params(axis='x', labelsize=25, labelrotation=45)
    ax.xaxis.set_major_formatter('{x:.0f}')
    ax.yaxis.set_major_formatter('{x:.0f}')

def apply_log2_xaxis(ax, x_order, left_pad=1.08, right_pad=1.18):
    """
    Match the reference spacing (8,16,32,...) using log base 2.
    Adds a bit of right padding so end-labels have room.
    """
    x_order = sorted(pd.unique(pd.Series(x_order).dropna()))
    ax.set_xscale("log", base=2)
    ax.set_xticks(x_order)

    x_set = set([float(x) for x in x_order])

    def _fmt(x, pos):
        if float(x) in x_set:
            if abs(x - round(x)) < 1e-9:
                return str(int(round(x)))
            return f"{x:g}"
        return ""

    ax.xaxis.set_major_formatter(ticker.FuncFormatter(_fmt))

    # Padding for labels on the right
    x_min = float(min(x_order))
    x_max = float(max(x_order))
    ax.set_xlim(x_min / left_pad, x_max * right_pad)


def _high_contrast_colors(n: int):
    """
    Use default Matplotlib colors first, then extend with Matplotlib categorical palettes.
    Designed for <= 13 models (your case), but remains robust if you exceed it.
    """
    n = int(n)
    if n <= 0:
        return []

    # 1) Default Matplotlib cycle (typically 10 colors: tab10)
    base = mpl.rcParams["axes.prop_cycle"].by_key().get("color", [])
    colors = [mpl.colors.to_hex(c) for c in base]

    if n <= len(colors):
        return colors[:n]

    # 2) Extend using additional built-in categorical palettes
    extra_sources = (
        list(colormaps["tab20"].colors)
        + list(colormaps["tab20b"].colors)
        + list(colormaps["tab20c"].colors)
    )

    seen = set(c.lower() for c in colors)
    for c in extra_sources:
        hx = mpl.colors.to_hex(c)
        if hx.lower() not in seen:
            colors.append(hx)
            seen.add(hx.lower())
        if len(colors) >= n:
            return colors[:n]

    # 3) Failsafe fallback (unlikely for your case)
    cmap = colormaps["turbo"]
    vals = np.linspace(0.0, 1.0, n, endpoint=True)
    return [mpl.colors.to_hex(cmap(v)) for v in vals]


def make_model_style_map(model_names):
    model_names = list(model_names)
    colors = _high_contrast_colors(len(model_names))

    markers = ["o", "s", "^", "v", "D", "P", "X", "<", ">", "h", "p", "8", "*"]
    dash_patterns = [
        None,          # solid
        (6, 2),        # dashed
        (2, 2),        # densely dashed
        (6, 2, 1, 2),  # dash-dot-ish
        (1, 1),        # dotted
        (3, 1, 1, 1),
    ]

    style_map = {}
    for i, m in enumerate(model_names):
        style_map[m] = {
            "color": colors[i],
            "marker": markers[i % len(markers)],
            "dashes": dash_patterns[i % len(dash_patterns)],
        }
    return style_map


def build_custom_lineplot(
    ax,
    data,
    style_map,
    highlight_models=None,
    highlight_kwargs=None,
    other_kwargs=None,
    model_col="model_raw_name",  # allows plotting by display-name column
):
    """
    Emphasis strategy:
      - non-highlight models: lower alpha, thinner lines, smaller markers
      - highlighted models: thicker lines, bigger markers, higher zorder (drawn on top)
    Returns:
      - end_points: list of dicts with x,y,color for labels (ONLY highlight models by default)
    """
    highlight_models = set(highlight_models or [])

    highlight_kwargs = highlight_kwargs or dict(
        linewidth=5,
        markersize=10.0,
        alpha=1.0,
        zorder=6,
        markeredgewidth=1.0,
        markeredgecolor="white",
    )
    other_kwargs = other_kwargs or dict(
        linewidth=4,
        markersize=10.0,
        alpha=1.0,
        zorder=2,
        markeredgewidth=0.0,
    )

    end_points = []

    models_in_data = list(pd.unique(data[model_col]))
    draw_order = [m for m in models_in_data if m not in highlight_models] + [
        m for m in models_in_data if m in highlight_models
    ]

    for model in draw_order:
        df_m = data[data[model_col] == model].sort_values("num_dims")
        if df_m.empty:
            continue

        st = style_map.get(model, {})
        is_highlight = model in highlight_models
        kw = highlight_kwargs if is_highlight else other_kwargs

        (line,) = ax.plot(
            df_m["num_dims"].to_numpy(),
            df_m["main_score"].to_numpy(),
            label=model,
            color=st.get("color", None),
            marker=st.get("marker", "o"),
            linewidth=kw["linewidth"],
            markersize=kw["markersize"],
            alpha=kw["alpha"],
            zorder=kw["zorder"],
            markeredgewidth=kw.get("markeredgewidth", 0.0),
            markeredgecolor=kw.get("markeredgecolor", None),
        )
        if st.get("dashes") is not None:
            line.set_dashes(st["dashes"])

        if is_highlight:
            last = df_m.iloc[-1]
            end_points.append(
                dict(
                    model=model,
                    x=float(last["num_dims"]),
                    y=float(last["main_score"]),
                    color=st.get("color", "black"),
                )
            )

    return end_points


def annotate_end_labels_with_vertical_spacing(
    ax,
    end_points,
    x_mult=1.06,
    min_sep_frac=0.03,
    clamp=True,
):
    """
    Places labels near the last point of each line, but "repels" them vertically
    so they don't overlap.
    """
    if not end_points:
        return

    y0, y1 = ax.get_ylim()
    y_span = (y1 - y0) if (y1 - y0) != 0 else 1.0
    min_sep = y_span * float(min_sep_frac)

    pts = sorted(end_points, key=lambda d: d["y"])
    y_adj = []
    for i, d in enumerate(pts):
        y_target = d["y"]
        if i == 0:
            y_new = y_target
        else:
            y_new = max(y_target, y_adj[-1] + min_sep)
        y_adj.append(y_new)

    if clamp:
        overflow = y_adj[-1] - y1
        if overflow > -0.02:
            y_adj = [yy - overflow for yy in y_adj]
        underflow = y0 - y_adj[0]
        if underflow > -0.02:
            y_adj = [yy + underflow for yy in y_adj]

    for d, yy in zip(pts, y_adj):
        x_text = d["x"] * float(x_mult)
        ax.annotate(
            d["model"],
            xy=(d["x"], d["y"]),
            xytext=(x_text, yy),
            textcoords="data",
            fontsize=17,
            fontweight="bold",
            color=d["color"],
            va="center",
            ha="left",
            zorder=10,
            arrowprops=dict(arrowstyle="-", lw=0.8, color=d["color"], alpha=0.7),
        )


def fix_legend_inside(ax, highlight_models=None, preset="bottom_right", ncols=2):
    """
    Keep legend inside axes. Reorder to show highlighted models first.
    Also supports multi-column legends with (near) equal column lengths.
    """
    highlight_models = set(highlight_models or [])
    handles, labels = ax.get_legend_handles_labels()

    # highlighted first
    idx_high = [i for i, lab in enumerate(labels) if lab in highlight_models]
    idx_rest = [i for i, lab in enumerate(labels) if lab not in highlight_models]
    new_idx = idx_high + idx_rest

    handles = [handles[i] for i in new_idx]
    labels  = [labels[i]  for i in new_idx]

    # --- ensure columns are equal length (only possible if even count)
    # If odd, add a dummy blank entry so each column has same number of rows.
    if ncols == 2 and (len(labels) % 2 == 1):
        handles.append(Line2D([], [], linestyle="none", marker=None, alpha=0.0))
        labels.append("")

    if preset == "upper_left":
        loc = "upper left"
        bbox = (0.02, 0.98)
    else:
        loc = "lower right"
        bbox = (0.98, 0.06)

    leg = ax.legend(
        handles,
        labels,
        loc=loc,
        bbox_to_anchor=bbox,
        frameon=True,
        framealpha=0.92,
        fontsize=12,
        borderaxespad=0.0,
        handlelength=3.0,
        labelspacing=0.35,
        ncol=ncols,              # <- key change (use ncols= in newer mpl)
        columnspacing=1.2,       # optional: spacing between the two columns
        handletextpad=0.6,       # optional: spacing between handle and text
    )
    leg.get_frame().set_linewidth(0.8)

# -------------------------
# Theme
# -------------------------
sns.set_theme(style="whitegrid", context="talk", font_scale=1.05)


# =========================
# 0) Model rename map
# =========================
MODEL_RENAME = {
    # Closed-source
    "global.cohere.embed-v4:0": "cohere-embed-v4",
    "amazon.titan-embed-text-v2:0": "amazon-titan-v2",
    "text-embedding-3-large": "text-embedding-3-large",
    # Peer-reviewed
    "serafim-900m-portuguese-pt-sentence-encoder": "serafim-900m",
    "multilingual-e5-large": "multilingual-e5-large",
    "Qwen3-Embedding-0.6B": "qwen3-embedding-0.6b",
    "gte-multilingual-base": "gte-multilingual-base",
    "Legal-BERTimbau-sts-large-ma-v3": "legal-bertimbau-large-sts",
    "BERTimbau-large": "bertimbau-large",
    "mmBERT-base": "mmbert-base",
    "ModBERT-BR": "modbertbr",
    "ModBERTBr": "modbertbr",
    # Community
    "NeoBERTugues": "neobertugues",
    "mmbert-embed-32k-2d-matryoshka": "mmbert-embed-32k",
    # Fine-tuned (Ours)
    "e5-large-matryoshka-sts-pt": "e5-large-matryoshka",
    "BERTimbau-large-matryoshka-sts-pt": "bertimbau-large-matryoshka",
    "ModBERTBr-matryoshka-sts-pt": "modbertbr-matryoshka",
}


# =========================
# 0.1) Per-task customizable y-bottom limits (NEW)
# =========================
# Use None to keep auto. Set any task you want to open space for legend.
Y_TOP_BY_TASK_NAME = {
    # Example:
    # "my_task_name": 0.40,
}

Y_TOP_BY_TASK_TYPE = {
    "STS": 89,
    "Classification": 57,
    "Retrieval": 82,
    "Reranking": 88,
}

Y_BOTTOM_BY_TASK_NAME = {
    # Example:
    # "my_task_name": 0.40,
}

Y_BOTTOM_BY_TASK_TYPE = {
    # Example:
    "STS": 67,
    "Classification": 40,
    "Retrieval": 0,
    "Reranking": 68
}

Y_BOTTOM_DEFAULT = None
Y_TOP_DEFAULT = None


# =========================
# 1) Preparação dos dados
# =========================
df_plot = results_all.copy()

df_plot["num_dims"] = df_plot["embedding_dim"].fillna(df_plot["embedding_dim_full"])
df_plot["num_dims"] = pd.to_numeric(df_plot["num_dims"], errors="coerce")
df_plot["main_score"] = pd.to_numeric(df_plot["main_score"], errors="coerce")

df_plot = df_plot.dropna(subset=["num_dims", "main_score", "model_raw_name"])
df_plot["model_raw_name"] = df_plot["model_raw_name"].astype(str)

# Display name used for legend + labels
df_plot["model_name"] = df_plot["model_raw_name"].replace(MODEL_RENAME)

# Build style map on display names (legend names)
model_order = sorted(df_plot["model_name"].unique())
MODEL_STYLE = make_model_style_map(model_order)

# Models to emphasize (and label at the end) — use DISPLAY NAMES
HIGHLIGHT_MODELS = {
    # Example:
    "bertimbau-large-matryoshka",
    "e5-large-matryoshka",
    "modbertbr-matryoshka",
}


# =========================================================
# 2) Gráficos por task_name
# =========================================================
task_names = sorted(df_plot["task_name"].dropna().unique())
score_map = {
    'Classification': 'Score',
    'STS': 'Score',
    'Retrieval': 'Score',
    'Reranking': 'Score'
}


for task in task_names:
    df_task = (
        df_plot[df_plot["task_name"] == task]
        .groupby(["task_name", "model_name", "num_dims"], as_index=False)["main_score"]
        .mean()
    )

    x_order = sorted(df_task["num_dims"].unique())

    fig, ax = plt.subplots(figsize=(11, 8))
    end_points = build_custom_lineplot(
        ax,
        df_task,
        MODEL_STYLE,
        highlight_models=HIGHLIGHT_MODELS,
        model_col="model_name",
    )

    ax.set_title(f"Task: {task}", fontsize=25, fontweight="bold", pad=10)
    ax.set_xlabel("Embedding Size", fontsize=20)
    ax.set_ylabel("Score", fontsize=20)

    apply_log2_xaxis(ax, x_order)

    y_override_bottom = Y_BOTTOM_BY_TASK_NAME.get(task, Y_BOTTOM_DEFAULT)
    y_override_top = Y_TOP_BY_TASK_NAME.get(task, Y_TOP_DEFAULT)
    apply_y_axis_formatting(
        ax,
        df_task["main_score"],
        step=0.03,
        margin=0.01,
        y_top_override=y_override_top,
        y_bottom_override=y_override_bottom,
    )

    annotate_end_labels_with_vertical_spacing(
        ax,
        end_points,
        x_mult=1.06,
        min_sep_frac=0.035,
        clamp=True,
    )

    # Keep the same behavior as before: bottom-right inside
    fix_legend_inside(ax, highlight_models=HIGHLIGHT_MODELS, preset="bottom_right")

    plt.tight_layout()

    filename = f"task_{sanitize_filename(task)}.pdf"
    fig.savefig(os.path.join(output_dir, filename), dpi=150, format="pdf")
    plt.close(fig)


# =========================================================
# 3) Gráficos por task_type (média)
# =========================================================
df_task_type = df_plot.groupby(["task_type", "model_name", "num_dims"], as_index=False)["main_score"].mean()

task_types = sorted(df_task_type["task_type"].dropna().unique())

for ttype in task_types:
    df_tt = df_task_type[df_task_type["task_type"] == ttype].copy()
    x_order = sorted(df_tt["num_dims"].unique())

    fig, ax = plt.subplots(figsize=(11, 8))
    end_points = build_custom_lineplot(
        ax,
        df_tt,
        MODEL_STYLE,
        highlight_models=HIGHLIGHT_MODELS,
        model_col="model_name",
    )

    # ax.set_title(f"Task Type: {ttype}", fontsize=30, fontweight="bold", pad=10)
    ax.set_xlabel("Embedding Size", fontsize=25)
    ax.set_ylabel(score_map[ttype], fontsize=25)

    apply_log2_xaxis(ax, x_order)

    y_override_bottom = Y_BOTTOM_BY_TASK_TYPE.get(ttype, Y_BOTTOM_DEFAULT)
    y_override_top = Y_TOP_BY_TASK_TYPE.get(ttype, Y_TOP_DEFAULT)

    apply_y_axis_formatting(
        ax,
        df_tt["main_score"],
        step=0.03,
        margin=0.01,
        y_top_override=y_override_top,
        y_bottom_override=y_override_bottom,
    )

    annotate_end_labels_with_vertical_spacing(
        ax,
        end_points,
        x_mult=1.06,
        min_sep_frac=0.035,
        clamp=True,
    )

    # NEW: Classification / Reranking -> upper-left inside; otherwise bottom-right inside
    legend_preset = "bottom_right"
    fix_legend_inside(ax, highlight_models=HIGHLIGHT_MODELS, preset=legend_preset)

    plt.tight_layout()

    filename = f"task_type_{sanitize_filename(ttype)}.pdf"
    fig.savefig(os.path.join(output_dir, filename), dpi=150, format="pdf")
    plt.close(fig)

### Graph Matryoshka Markers

In [ ]:
# # -------------------------
# # Utils
# # -------------------------
# def sanitize_filename(name):
#     name = str(name)
#     name = re.sub(r"[^\w\s-]", "", name)
#     name = re.sub(r"\s+", "_", name)
#     return name.lower()


# def apply_y_axis_formatting(
#     ax,
#     series,
#     step=0.03,
#     margin=0.01,
#     y_bottom_override=None,
#     y_top_override=None,
# ):
#     y_min = float(series.min())
#     y_max = float(series.max())

#     y_bottom = (np.floor(y_min / step) * step) - margin
#     y_top = (np.floor(y_max / step) * step) + step + margin

#     if y_bottom_override is not None:
#         y_bottom = float(y_bottom_override)

#     if y_top_override is not None:
#         y_top = float(y_top_override)

#     ax.set_ylim(y_bottom, y_top)
#     ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
#     ax.tick_params(axis="y", labelsize=25)
#     ax.tick_params(axis="x", labelsize=25, labelrotation=45)


# def apply_log2_xaxis(ax, x_order, left_pad=1.08, right_pad=1.18):
#     x_order = sorted(pd.unique(pd.Series(x_order).dropna()))
#     ax.set_xscale("log", base=2)
#     ax.set_xticks(x_order)

#     x_set = set([float(x) for x in x_order])

#     def _fmt(x, pos):
#         if float(x) in x_set:
#             if abs(x - round(x)) < 1e-9:
#                 return str(int(round(x)))
#             return f"{x:g}"
#         return ""

#     ax.xaxis.set_major_formatter(ticker.FuncFormatter(_fmt))

#     x_min = float(min(x_order))
#     x_max = float(max(x_order))
#     ax.set_xlim(x_min / left_pad, x_max * right_pad)


# def _high_contrast_colors(n: int):
#     n = int(n)
#     if n <= 0:
#         return []

#     base = mpl.rcParams["axes.prop_cycle"].by_key().get("color", [])
#     colors = [mpl.colors.to_hex(c) for c in base]

#     if n <= len(colors):
#         return colors[:n]

#     extra_sources = (
#         list(colormaps["tab20"].colors)
#         + list(colormaps["tab20b"].colors)
#         + list(colormaps["tab20c"].colors)
#     )

#     seen = set(c.lower() for c in colors)
#     for c in extra_sources:
#         hx = mpl.colors.to_hex(c)
#         if hx.lower() not in seen:
#             colors.append(hx)
#             seen.add(hx.lower())
#         if len(colors) >= n:
#             return colors[:n]

#     cmap = colormaps["turbo"]
#     vals = np.linspace(0.0, 1.0, n, endpoint=True)
#     return [mpl.colors.to_hex(cmap(v)) for v in vals]


# # -------------------------
# # NEW: group helpers
# # -------------------------
# def extract_first_substring(model_name: str) -> str:
#     """
#     Returns first token before '-'.
#     Example:
#       'e5-large-matryoshka-sts-pt' -> 'e5'
#       'BERTimbau-large-sts-pt' -> 'BERTimbau'
#     """
#     model_name = str(model_name)
#     return model_name.split("-")[0].strip()


# def infer_model_group(model_name: str, group_substrings=None) -> str:
#     """
#     Assign model to a group based on the first substring before '-'
#     and an allowlist of substrings.

#     If first substring is in group_substrings, use it as group.
#     Otherwise, fall back to the full model name, so unrelated models
#     keep independent colors.
#     """
#     first = extract_first_substring(model_name)
#     group_substrings = set(group_substrings or [])
#     if first in group_substrings:
#         return first
#     return str(model_name)


# def infer_model_variant(model_name: str) -> str:
#     """
#     Variant inside a group.
#     Used to assign line patterns while preserving group color.
#     """
#     model_name = str(model_name).lower()
#     if "matryoshka" in model_name:
#         return "matryoshka"
#     return "default"


# def make_model_style_map_grouped(model_names, group_substrings=None):
#     """
#     Build style map so models in the same group share color,
#     while variants inside the group get different dash patterns.

#     Grouping rule:
#       - use first substring before '-'
#       - only substrings in group_substrings are treated as shared groups
#       - others get their own independent group/color
#     """
#     model_names = list(model_names)
#     group_substrings = list(group_substrings or [])

#     # Determine group per model
#     model_to_group = {
#         m: infer_model_group(m, group_substrings=group_substrings)
#         for m in model_names
#     }

#     unique_groups = list(dict.fromkeys(model_to_group[m] for m in model_names))
#     group_colors = _high_contrast_colors(len(unique_groups))
#     group_to_color = {g: c for g, c in zip(unique_groups, group_colors)}

#     # Variants within each group get line patterns
#     variant_dash_map = {
#         "default": None,     # solid
#         "matryoshka": (2, 1) # dashed
#     }

#     # Optional: markers also differ a bit by variant
#     variant_marker_map = {
#         "default": "o",
#         "matryoshka": "s",
#     }

#     style_map = {}
#     for m in model_names:
#         group = model_to_group[m]
#         variant = infer_model_variant(m)

#         style_map[m] = {
#             "group": group,
#             "variant": variant,
#             "color": group_to_color[group],
#             "marker": variant_marker_map.get(variant, "o"),
#             "dashes": variant_dash_map.get(variant, None),
#         }

#     return style_map


# def build_custom_lineplot(
#     ax,
#     data,
#     style_map,
#     highlight_models=None,
#     highlight_kwargs=None,
#     other_kwargs=None,
#     model_col="model_raw_name",
# ):
#     highlight_models = set(highlight_models or [])

#     highlight_kwargs = highlight_kwargs or dict(
#         linewidth=5,
#         markersize=10.0,
#         alpha=1.0,
#         zorder=6,
#         markeredgewidth=1.0,
#         markeredgecolor="white",
#     )
#     other_kwargs = other_kwargs or dict(
#         linewidth=4,
#         markersize=10.0,
#         alpha=1.0,
#         zorder=2,
#         markeredgewidth=0.0,
#     )

#     end_points = []

#     models_in_data = list(pd.unique(data[model_col]))
#     draw_order = [m for m in models_in_data if m not in highlight_models] + [
#         m for m in models_in_data if m in highlight_models
#     ]

#     for model in draw_order:
#         df_m = data[data[model_col] == model].sort_values("num_dims")
#         if df_m.empty:
#             continue

#         st = style_map.get(model, {})
#         is_highlight = model in highlight_models
#         kw = highlight_kwargs if is_highlight else other_kwargs

#         (line,) = ax.plot(
#             df_m["num_dims"].to_numpy(),
#             df_m["main_score"].to_numpy(),
#             label=model,
#             color=st.get("color", None),
#             marker=st.get("marker", "o"),
#             linewidth=kw["linewidth"],
#             markersize=kw["markersize"],
#             alpha=kw["alpha"],
#             zorder=kw["zorder"],
#             markeredgewidth=kw.get("markeredgewidth", 0.0),
#             markeredgecolor=kw.get("markeredgecolor", None),
#         )
#         if st.get("dashes") is not None:
#             line.set_dashes(st["dashes"])

#         if is_highlight:
#             last = df_m.iloc[-1]
#             end_points.append(
#                 dict(
#                     model=model,
#                     x=float(last["num_dims"]),
#                     y=float(last["main_score"]),
#                     color=st.get("color", "black"),
#                 )
#             )

#     return end_points


# def annotate_end_labels_with_vertical_spacing(
#     ax,
#     end_points,
#     x_mult=1.06,
#     min_sep_frac=0.03,
#     clamp=True,
# ):
#     if not end_points:
#         return

#     y0, y1 = ax.get_ylim()
#     y_span = (y1 - y0) if (y1 - y0) != 0 else 1.0
#     min_sep = y_span * float(min_sep_frac)

#     pts = sorted(end_points, key=lambda d: d["y"])
#     y_adj = []
#     for i, d in enumerate(pts):
#         y_target = d["y"]
#         if i == 0:
#             y_new = y_target
#         else:
#             y_new = max(y_target, y_adj[-1] + min_sep)
#         y_adj.append(y_new)

#     if clamp:
#         overflow = y_adj[-1] - y1
#         if overflow > 0:
#             y_adj = [yy - overflow for yy in y_adj]
#         underflow = y0 - y_adj[0]
#         if underflow > 0:
#             y_adj = [yy + underflow for yy in y_adj]

#     for d, yy in zip(pts, y_adj):
#         x_text = d["x"] * float(x_mult)
#         ax.annotate(
#             d["model"],
#             xy=(d["x"], d["y"]),
#             xytext=(x_text, yy),
#             textcoords="data",
#             fontsize=17,
#             fontweight="bold",
#             color=d["color"],
#             va="center",
#             ha="left",
#             zorder=10,
#             arrowprops=dict(arrowstyle="-", lw=0.8, color=d["color"], alpha=0.7),
#         )


# def fix_legend_inside(ax, highlight_models=None, preset="bottom_right"):
#     highlight_models = set(highlight_models or [])
#     handles, labels = ax.get_legend_handles_labels()

#     idx_high = [i for i, lab in enumerate(labels) if lab in highlight_models]
#     idx_rest = [i for i, lab in enumerate(labels) if lab not in highlight_models]
#     new_idx = idx_high + idx_rest

#     handles = [handles[i] for i in new_idx]
#     labels = [labels[i] for i in new_idx]

#     if preset == "upper_left":
#         loc = "upper left"
#         bbox = (0.02, 0.98)
#     else:
#         loc = "lower right"
#         bbox = (0.98, 0.06)

#     leg = ax.legend(
#         handles,
#         labels,
#         loc=loc,
#         bbox_to_anchor=bbox,
#         frameon=True,
#         framealpha=0.92,
#         fontsize=12,
#         borderaxespad=0.0,
#         handlelength=3.0,
#         labelspacing=0.35
#     )
#     leg.get_frame().set_linewidth(0.8)


# # -------------------------
# # Theme
# # -------------------------
# sns.set_theme(style="whitegrid", context="talk", font_scale=1.05)


# # =========================
# # 0) Model rename map
# # =========================
# MODEL_RENAME = {
#     "global.cohere.embed-v4:0": "Cohere Embed v4",
#     "amazon.titan-embed-text-v2:0": "Amazon Titan v2",
#     "text-embedding-3-large": "Text Embedding 3 Large",
#     "serafim-900m-portuguese-pt-sentence-encoder": "Serafim 900m",
#     "multilingual-e5-large": "Multilingual e5 Large",
#     "Qwen3-Embedding-0.6B": "Qwen3 Embedding 0.6B",
#     "qwen3-embedding-0.6b": "Qwen3 Embedding 0.6B",
#     "gte-multilingual-base": "GTE Multilingual Base",
#     "Legal-BERTimbau-sts-large-ma-v3": "Legal BERTimbau STS",
#     "BERTimbau-large": "BERTimbau Large",
#     "mmBERT-base": "mmBERT Base",
#     "ModBERT-BR": "ModBERTBr",
#     "ModBERTBr": "ModBERTBr",
#     "NeoBERTugues": "NeoBERTugues",
#     "mmbert-embed-32k-2d-matryoshka": "mmBERT Embed 32k",
#     "e5-large-matryoshka-sts-pt": "e5 Large MRL",
#     "BERTimbau-large-matryoshka-sts-pt": "BERTimbau Large MRL",
#     "ModBERTBr-matryoshka-sts-pt": "ModBERTBr MRL",
#     "e5-large-sts-pt": "e5 Large w/o MRL",
#     "BERTimbau-large-sts-pt": "BERTimbau Large w/o MRL",
#     "ModBERTBr-sts-pt": "ModBERTBr w/o MRL",
# }


# # =========================
# # NEW: model grouping substrings
# # =========================
# GROUP_SUBSTRINGS = [
#     "e5",
#     "BERTimbau",
#     "ModBERTBr",
# ]


# # =========================
# # 0.1) Per-task customizable y-bottom limits
# # =========================
# Y_TOP_BY_TASK_NAME = {}
# Y_TOP_BY_TASK_TYPE = {}
# Y_BOTTOM_BY_TASK_NAME = {}
# Y_BOTTOM_BY_TASK_TYPE = {}

# Y_BOTTOM_DEFAULT = None
# Y_TOP_DEFAULT = None


# # =========================
# # 1) Preparação dos dados
# # =========================
# df_plot = results_all.copy()

# df_plot["num_dims"] = df_plot["embedding_dim"].fillna(df_plot["embedding_dim_full"])
# df_plot["num_dims"] = pd.to_numeric(df_plot["num_dims"], errors="coerce")
# df_plot["main_score"] = pd.to_numeric(df_plot["main_score"], errors="coerce")

# df_plot = df_plot.dropna(subset=["num_dims", "main_score", "model_raw_name"])
# df_plot["model_raw_name"] = df_plot["model_raw_name"].astype(str)

# # Display name used for legend + labels
# df_plot["model_name"] = df_plot["model_raw_name"].replace(MODEL_RENAME)

# # NEW: keep raw grouping signal, but style the displayed names
# # group is inferred from raw model name, not display name
# display_to_raw = (
#     df_plot[["model_name", "model_raw_name"]]
#     .drop_duplicates()
#     .set_index("model_name")["model_raw_name"]
#     .to_dict()
# )

# model_order = sorted(df_plot["model_name"].unique())

# MODEL_STYLE = make_model_style_map_grouped(
#     model_names=model_order,
#     group_substrings=GROUP_SUBSTRINGS,
# )

# # Recompute styles using raw names for grouping, but store keyed by display name
# MODEL_STYLE = {}
# base_style_map = make_model_style_map_grouped(
#     model_names=df_plot["model_raw_name"].unique(),
#     group_substrings=GROUP_SUBSTRINGS,
# )

# for display_name in model_order:
#     raw_name = display_to_raw[display_name]
#     MODEL_STYLE[display_name] = base_style_map[raw_name]

# # Models to emphasize (DISPLAY NAMES)
# HIGHLIGHT_MODELS = {
#     "BERTimbau Large MRL",
#     "e5 Large MRL",
#     "ModBERTBr MRL",
# }


# # =========================================================
# # 2) Gráficos por task_name
# # =========================================================
# task_names = sorted(df_plot["task_name"].dropna().unique())
# score_map = {
#     "Classification": "Score",
#     "STS": "Score",
#     "Retrieval": "Score",
#     "Reranking": "Score"
# }

# for task in task_names:
#     df_task = (
#         df_plot[df_plot["task_name"] == task]
#         .groupby(["task_name", "model_name", "num_dims"], as_index=False)["main_score"]
#         .mean()
#     )

#     x_order = sorted(df_task["num_dims"].unique())

#     fig, ax = plt.subplots(figsize=(11, 8))
#     end_points = build_custom_lineplot(
#         ax,
#         df_task,
#         MODEL_STYLE,
#         highlight_models=HIGHLIGHT_MODELS,
#         model_col="model_name",
#     )

#     ax.set_title(f"Task: {task}", fontsize=25, fontweight="bold", pad=10)
#     ax.set_xlabel("Embedding Size", fontsize=20)
#     ax.set_ylabel("Score", fontsize=20)

#     apply_log2_xaxis(ax, x_order)

#     y_override_bottom = Y_BOTTOM_BY_TASK_NAME.get(task, Y_BOTTOM_DEFAULT)
#     y_override_top = Y_TOP_BY_TASK_NAME.get(task, Y_TOP_DEFAULT)
#     apply_y_axis_formatting(
#         ax,
#         df_task["main_score"],
#         step=0.03,
#         margin=0.01,
#         y_top_override=y_override_top,
#         y_bottom_override=y_override_bottom,
#     )

#     annotate_end_labels_with_vertical_spacing(
#         ax,
#         end_points,
#         x_mult=1.06,
#         min_sep_frac=0.035,
#         clamp=True,
#     )

#     fix_legend_inside(ax, highlight_models=HIGHLIGHT_MODELS, preset="bottom_right")

#     plt.tight_layout()

#     filename = f"task_{sanitize_filename(task)}.pdf"
#     fig.savefig(os.path.join(output_dir, filename), dpi=150, format="pdf")
#     plt.close(fig)


# # =========================================================
# # 3) Gráficos por task_type (média)
# # =========================================================
# df_task_type = (
#     df_plot.groupby(["task_type", "model_name", "num_dims"], as_index=False)["main_score"]
#     .mean()
# )

# task_types = sorted(df_task_type["task_type"].dropna().unique())

# for ttype in task_types:
#     df_tt = df_task_type[df_task_type["task_type"] == ttype].copy()
#     x_order = sorted(df_tt["num_dims"].unique())

#     fig, ax = plt.subplots(figsize=(11, 8))
#     end_points = build_custom_lineplot(
#         ax,
#         df_tt,
#         MODEL_STYLE,
#         highlight_models=HIGHLIGHT_MODELS,
#         model_col="model_name",
#     )

#     ax.set_xlabel("Embedding Size", fontsize=25)
#     ax.set_ylabel(score_map[ttype], fontsize=25)

#     apply_log2_xaxis(ax, x_order)

#     y_override_bottom = Y_BOTTOM_BY_TASK_TYPE.get(ttype, Y_BOTTOM_DEFAULT)
#     y_override_top = Y_TOP_BY_TASK_TYPE.get(ttype, Y_TOP_DEFAULT)

#     apply_y_axis_formatting(
#         ax,
#         df_tt["main_score"],
#         step=0.03,
#         margin=0.01,
#         y_top_override=y_override_top,
#         y_bottom_override=y_override_bottom,
#     )

#     annotate_end_labels_with_vertical_spacing(
#         ax,
#         end_points,
#         x_mult=1.06,
#         min_sep_frac=0.035,
#         clamp=True,
#     )

#     legend_preset = "bottom_right"
#     fix_legend_inside(ax, highlight_models=HIGHLIGHT_MODELS, preset=legend_preset)

#     plt.tight_layout()

#     filename = f"task_type_{sanitize_filename(ttype)}.pdf"
#     fig.savefig(os.path.join(output_dir, filename), dpi=150, format="pdf")
#     plt.close(fig)

## Ratio Metrics

In [17]:
# =========================================================
# 5) Ratio vs full embedding por task_type
# =========================================================

map_lim_task = {
    "Classification": 0.8,
    "Reranking": 0.85,
    "Retrieval": 0.35,
    "STS": 0.9,
}

MODEL_RENAME = {
    'e5-large-matryoshka-sts-pt': 'e5 Large Matryoshka STS',
    'e5-large-sts-pt': 'e5 Large STS',
    'BERTimbau-large-matryoshka-sts-pt': 'BERTimbau Large Matryoshka STS',
    'BERTimbau-large-sts-pt': 'BERTimbau Large STS',
    'ModBERTBr-matryoshka-sts-pt': 'ModBERTBr Large Matryoshka STS',
    'ModBERTBr-sts-pt': 'ModBERTBr Large STS'
}

df_ratio = df_plot.copy()

df_ratio["num_dims"] = pd.to_numeric(df_ratio["num_dims"], errors="coerce").astype("Int64")
df_ratio["main_score"] = pd.to_numeric(df_ratio["main_score"], errors="coerce")

df_ratio = df_ratio.dropna(subset=["task_type", "model_raw_name", "num_dims", "main_score"])
df_ratio["num_dims"] = df_ratio["num_dims"].astype(int)

df_ratio = (
    df_ratio.groupby(["task_type", "model_raw_name", "num_dims"], as_index=False)["main_score"]
    .mean()
)


def get_hue_order(models):
    """
    Retorna a lista de modelos ordenada: primeiro os sem matryoshka,
    depois os com matryoshka — agrupando visualmente as barras.
    """
    standard = sorted([m for m in models if ("matryoshka" not in m.lower() and "qwen" not in m.lower())])
    matryoshka = sorted([m for m in models if ("matryoshka" in m.lower() or "qwen" in m.lower())])
    return standard + matryoshka


def apply_bar_hatch(ax):
    """
    Aplica hachura nos modelos com 'matryoshka' (ou 'qwen') e preenchimento
    sólido nos demais. Cria duas legendas no canto superior direito, dentro do gráfico.
    """
    handles, labels = ax.get_legend_handles_labels()

    # # Aplica hatch nas barras (cada container corresponde a um item do hue)
    # for container, label in zip(ax.containers, labels):
    #     hatch = "//" if ("matryoshka" in label.lower() or "qwen" in label.lower()) else ""
    #     for bar in container:
    #         bar.set_hatch(hatch)

    # # Espelha o hatch nos itens da legenda dos modelos
    # for handle, label in zip(handles, labels):
    #     hatch = "//" if ("matryoshka" in label.lower() or "qwen" in label.lower()) else ""
    #     try:
    #         handle.set_hatch(hatch)
    #     except Exception:
    #         pass

    # Remove qualquer legenda padrão criada pelo seaborn (vamos recriar)
    if ax.get_legend() is not None:
        ax.get_legend().remove()

    # --- Bloco 1: legenda dos modelos (cores + hatch), top-right (inside) ---
    leg1 = ax.legend(
        handles=handles,
        labels=labels,
        loc="lower right",
        fontsize=12,
        frameon=True,
        borderaxespad=0.4,
    )
    ax.add_artist(leg1)

    ax.tick_params(axis="y", labelsize=25)
    ax.tick_params(axis='x', labelsize=25, labelrotation=45)

    # # --- Bloco 2: legenda do tipo de preenchimento (abaixo), ainda top-right (inside) ---
    # solid_patch = mpatches.Patch(facecolor="white", edgecolor="black", hatch="", label="Standard")
    # hatched_patch = mpatches.Patch(facecolor="white", edgecolor="black", hatch="//", label="Matryoshka")

    # ax.legend(
    #     handles=[solid_patch, hatched_patch],
    #     title="Loss",
    #     loc="upper right",
    #     bbox_to_anchor=(1.0, 0.74),   # ajuste fino para ficar abaixo da leg1 (dentro do axes)
    #     bbox_transform=ax.transAxes,
    #     fontsize=9,
    #     title_fontsize=10,
    #     frameon=True,
    #     borderaxespad=0.4,
    # )


task_types = sorted(df_ratio["task_type"].dropna().unique())

for ttype in task_types:
    df_tt = df_ratio[df_ratio["task_type"] == ttype].copy()

    max_dims = (
        df_tt.groupby("model_raw_name")["num_dims"]
        .max()
        .rename("max_dim")
        .reset_index()
    )
    df_tt = df_tt.merge(max_dims, on="model_raw_name", how="left")

    df_max = (
        df_tt[df_tt["num_dims"] == df_tt["max_dim"]][["model_raw_name", "main_score"]]
        .rename(columns={"main_score": "max_score"})
    )

    df_tt = df_tt.merge(df_max, on="model_raw_name", how="left")
    df_tt["ratio"] = df_tt["main_score"] / df_tt["max_score"]
    df_tt['model_raw_name'] = df_tt['model_raw_name'].map(MODEL_RENAME)

    hue_order = get_hue_order(df_tt["model_raw_name"].unique())

    fig, ax = plt.subplots(figsize=(5, 8))

    sns.barplot(
        data=df_tt,
        x="num_dims",
        y="ratio",
        hue="model_raw_name",
        hue_order=hue_order,
        ax=ax,
    )

    # ax.set_title(f"Performance Ratio - Task Type: {ttype}")
    ax.set_xlabel("Embedding Size", fontsize=25)
    ax.set_ylabel("Ratio of maximum performance", fontsize=25)
    ax.set_ylim(map_lim_task.get(ttype, 0.0), 1.0)

    apply_bar_hatch(ax)

    plt.tight_layout()

    filename = f"ratio_task_type_{sanitize_filename(ttype)}.pdf"
    fig.savefig(os.path.join(output_dir, filename), dpi=150, format="pdf")
    plt.close(fig)

AttributeError: 'float' object has no attribute 'lower'

## Global Metrics

In [19]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LogNorm


# -----------------------------
# formatting helpers
# -----------------------------
def _human_suffix(n: float) -> str:
    n = float(n)
    abs_n = abs(n)

    def _trim(x: float) -> str:
        return f"{x:.3f}".rstrip("0").rstrip(".")

    if abs_n >= 1e12:
        return _trim(n / 1e12) + "T"
    if abs_n >= 1e9:
        return _trim(n / 1e9) + "B"
    if abs_n >= 1e6:
        return _trim(n / 1e6) + "M"
    if abs_n >= 1e3:
        return _trim(n / 1e3) + "K"
    return _trim(n)


def _log_tick_formatter(x, pos=None):
    if x <= 0:
        return ""

    exp = np.floor(np.log10(x))
    mant = x / (10**exp)

    if np.isclose(mant, 1.0, rtol=0, atol=1e-10):
        return _human_suffix(x)
    if np.isclose(mant, 2.0, rtol=0, atol=1e-10):
        return "2"
    if np.isclose(mant, 5.0, rtol=0, atol=1e-10):
        return "5"
    return ""


def _tokens_tick_formatter(v, pos=None):
    return _human_suffix(v)


def _move_text_by_points(ax, text_obj, dx_pt: float, dy_pt: float):
    """
    Move a Text object by (dx, dy) given in points.
    Uses display coords under the hood, then maps back to data coords.
    """
    # points -> pixels
    dx_px = dx_pt * ax.figure.dpi / 72.0
    dy_px = dy_pt * ax.figure.dpi / 72.0

    x_data, y_data = text_obj.get_position()
    x_disp, y_disp = ax.transData.transform((x_data, y_data))
    new_disp = (x_disp + dx_px, y_disp + dy_px)
    new_data = ax.transData.inverted().transform(new_disp)
    text_obj.set_position(tuple(new_data))


# -----------------------------
# main plot
# -----------------------------
def plot_modelsize_vs_score(
    df: pd.DataFrame,
    output_path: str = "model_size_vs_main_score.pdf",
    annotate: bool = True,
    cmap: str = "plasma",
    fontsize: int = 9,
    label_offsets: dict | None = None,  # {"model_raw_name": (dx_pt, dy_pt)}
    label_align: dict | None = None,    # {"model_raw_name": {"ha":"left","va":"top"}}
    draw_arrows: bool = False,
):
    """
    Manual label positioning only (no auto-repel).

    label_offsets:
      dict mapping model_raw_name -> (dx_pt, dy_pt) in points

    label_align:
      dict mapping model_raw_name -> kwargs for alignment, e.g. {"ha":"left","va":"center"}

    draw_arrows:
      if True, draws a connector line from text to point (helps when offsets are large)
    """
    required = {"model_raw_name", "model_size", "main_score", "max_token"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in df: {sorted(missing)}")

    if label_offsets is None:
        label_offsets = {}
    if label_align is None:
        label_align = {}

    data = df.copy()
    data["model_size"] = pd.to_numeric(data["model_size"], errors="coerce")
    data["main_score"] = pd.to_numeric(data["main_score"], errors="coerce")
    data["max_token"] = pd.to_numeric(data["max_token"], errors="coerce")

    data = data.dropna(subset=["model_raw_name", "model_size", "main_score", "max_token"])
    data = data[(data["model_size"] > 0) & (data["max_token"] > 0)]

    # aggregate: mean score, max tokens
    agg = (
        data.groupby(["model_raw_name", "model_size"], as_index=False)
        .agg(
            main_score_avg=("main_score", "mean"),
            max_token=("max_token", "max"),
        )
        .sort_values("model_size")
    )

    # dot sizes (scaled but monotonic in model_size)
    ms = agg["model_size"].to_numpy(dtype=float)
    ms_min, ms_max = ms.min(), ms.max()
    s_min, s_max = 40, 750
    sizes = s_min + (np.sqrt(ms) - np.sqrt(ms_min)) * (s_max - s_min) / (
        np.sqrt(ms_max) - np.sqrt(ms_min) + 1e-12
    )

    # colors: max_token with log norm
    tokens = agg["max_token"].to_numpy(dtype=float)
    norm = LogNorm(vmin=tokens.min(), vmax=tokens.max())

    fig, ax = plt.subplots(figsize=(14, 5.8))

    sc = ax.scatter(
        agg["model_size"],
        agg["main_score_avg"],
        s=sizes,
        c=tokens,
        cmap=cmap,
        norm=norm,
        alpha=0.82,
        edgecolor="white",
        linewidth=0.8,
        zorder=2,
    )

    ax.set_xscale("log")
    ax.grid(True, which="both", linestyle="-", linewidth=0.4, alpha=0.35, zorder=1)
    ax.set_xlabel("Number of Parameters", fontsize=20)
    ax.set_ylabel("Mean (Task)", fontsize=20)

    # X ticks formatting (use mticker consistently)
    ax.xaxis.set_major_locator(mticker.LogLocator(base=10, subs=(1.0,)))
    ax.xaxis.set_minor_locator(mticker.LogLocator(base=10, subs=(2.0, 5.0)))
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(_log_tick_formatter))
    ax.xaxis.set_minor_formatter(mticker.FuncFormatter(_log_tick_formatter))
    ax.tick_params(axis="y", labelsize=20)
    ax.tick_params(axis="x", which="minor", labelsize=15, pad=6)
    ax.tick_params(axis="x", which="major", labelsize=20, pad=8)

    # Colorbar
    cbar = fig.colorbar(sc, ax=ax, pad=0.02)
    cbar.set_label("Max Tokens", fontsize=20)
    
    # --- Colorbar ticks: log-spaced with intermediate ticks + include min/max ---
    tmin, tmax = float(tokens.min()), float(tokens.max())

    # Build candidate ticks at 1, 2, 5 for each decade (classic log tick pattern)
    locator = mticker.LogLocator(base=10, subs=(1.0, 2.0, 5.0), numticks=100)
    ticks = locator.tick_values(tmin, tmax)

    # Keep only ticks inside [tmin, tmax], and force-add endpoints
    ticks = ticks[(ticks >= tmin) & (ticks <= tmax)]
    ticks = np.unique(np.r_[tmin, ticks, tmax])   # add min/max and dedupe
    ticks = np.sort(ticks)

    cbar.set_ticks(ticks)
    cbar.ax.yaxis.set_major_formatter(mticker.FuncFormatter(_tokens_tick_formatter))

    # margins
    xmin, xmax = agg["model_size"].min(), agg["model_size"].max()
    ax.set_xlim(xmin / 1.3, xmax * 1.35)

    ymin, ymax = agg["main_score_avg"].min(), agg["main_score_avg"].max()
    ypad = (ymax - ymin) * 0.22 if ymax > ymin else 0.02
    ax.set_ylim(ymin - 0.15 * ypad, ymax + ypad)

    # annotations (manual offsets only)
    if annotate:
        default_offset = (0.0, 8.0)  # in points
        default_align = {"ha": "center", "va": "bottom"}
        arrowprops = dict(arrowstyle="-", lw=0.6, color="black", alpha=0.45) if draw_arrows else None

        for _, r in agg.iterrows():
            name = str(r["model_raw_name"])
            x = float(r["model_size"])
            y = float(r["main_score_avg"])

            align = dict(default_align)
            align.update(label_align.get(name, {}))

            # place text at the point (data coords), then shift by points via label_offsets
            t = ax.text(x, y, name, fontsize=fontsize, alpha=0.95, zorder=3, **align)

            dx_pt, dy_pt = label_offsets.get(name, default_offset)
            _move_text_by_points(ax, t, dx_pt, dy_pt)

            # optional connector line to the dot
            if draw_arrows:
                ax.annotate(
                    "",
                    xy=(x, y),
                    xytext=t.get_position(),
                    textcoords="data",
                    arrowprops=arrowprops,
                    zorder=2.5,
                )

    plt.tight_layout()

    ext = os.path.splitext(output_path)[1].lower()
    if ext in [".png", ".jpg", ".jpeg", ".tif", ".tiff"]:
        plt.savefig(output_path, dpi=220, bbox_inches="tight")
    else:
        plt.savefig(output_path, bbox_inches="tight")

    plt.close(fig)
    return agg


# -----------------------------
# Example usage
# -----------------------------
filename = "global_results.pdf"
filepath = os.path.join(output_dir, filename)

LABEL_OFFSETS = {
    "Qwen3-Embedding-0.6B": (60, 0),
    "serafim-900m-portuguese-pt-sentence-encoder": (-10, -25),
    "e5-large-matryoshka-sts-pt": (0, 15),
    "multilingual-e5-large": (0, -20),
}
LABEL_ALIGN = {
    # "serafim-900m-portuguese-pt-sentence-encoder": {"ha": "left", "va": "center"},
}

agg_df = plot_modelsize_vs_score(
    results_all.query("embedding_dim_full==max_emb_size"),
    output_path=filepath,
    label_offsets=LABEL_OFFSETS,
    label_align=LABEL_ALIGN,
    draw_arrows=True,   # handy when offsets are large
)
print(agg_df.head())

                           model_raw_name   model_size  main_score_avg  \
10                  multilingual-e5-small  100000000.0        0.671162   
6          mmbert-embed-32k-2d-matryoshka  110000000.0        0.661942   
2             ModBERTBr-matryoshka-sts-pt  149000000.0        0.642619   
7                    multilingual-e5-base  270000000.0        0.673260   
11  paraphrase-multilingual-mpnet-base-v2  300000000.0        0.647503   

    max_token  
10      512.0  
6     32000.0  
2       512.0  
7       512.0  
11      128.0  
